In [44]:
from WindPy import w
import numpy as np
import pandas as pd
import datetime
import os
import json
from tqdm import tqdm

w.start()
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2024 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [43]:
w.close()

In [64]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)

In [37]:
df = pd.read_csv("A股指数.csv")
a_share_dict = dict(zip(df["Ticker"], df["Name"]))

df = pd.read_csv("海外指数.csv")
other_market_dict = dict(zip(df["Ticker"], df["Name"]))

print(len(a_share_dict), len(other_market_dict))

333 57


In [65]:
len(INDEX_START)

203

# Load Local Database

In [39]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

186

In [40]:
print(INDEX_DATA['000300.SH'].index[-1])

2025-05-23


# Update Existing Tickers

In [58]:
today = "2025-05-28"
today_date = datetime.datetime.strptime(today, "%Y-%m-%d").date()
today_date

datetime.date(2025, 5, 28)

In [59]:
test_start = "2025-05-23"
temp = w.wsd('000300.SH', 'pe_ttm,dividendyield2,close', test_start, today, '', usedf = True)[1]
temp.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-23,12.5468,3.4490,3882.2741
2025-05-26,12.4877,3.4660,3860.1067
2025-05-27,12.4516,3.4803,3839.3969
2025-05-28,12.4429,3.4760,3836.2366


In [32]:
# for ticker in tqdm(INDEX_DATA):
#     data = INDEX_DATA[ticker]

#     idx = -1
#     last = date.index[idx]
#     while not (type(last) is datetime.date):
#         idx -= 1
#         last = date.index[idx]

#     start = (last + pd.Timedelta(days = 1)).strftime("%Y-%m-%d")
    
#     new_data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]

#     if len(new_data) == 1 and new_data.index[0] == ticker:
#         new_data.index = [today_date]
    
#     if new_data.columns[0] == 'OUTMESSAGE':
#         print(ticker)
#         continue
    
#     INDEX_DATA[ticker] = pd.concat([data, new_data])
#     INDEX_DATA[ticker].dropna(inplace = True)

In [60]:
for ticker in tqdm(INDEX_DATA):
    data = INDEX_DATA[ticker]

    if data.index[-1] == today_date:
        continue
    
    start = data.index[-1].strftime("%Y-%m-%d")

    new_data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]

    data = pd.concat([data, new_data]).groupby(level = 0).last()
    data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
    data.index = data.index.date

    data = data[~data.index.duplicated(keep = "last")]

    # if len(new_data) == 1 and new_data.index[0] == ticker:
    #     new_data.index = [today_date]
    
    if new_data.columns[0] == 'OUTMESSAGE':
        print(ticker)
        continue

    INDEX_DATA[ticker] = data.copy(deep = True)
    INDEX_DATA[ticker].dropna(inplace = True)

100%|████████████████████████████████████████████████████████████████████████████████| 186/186 [01:45<00:00,  1.77it/s]


In [53]:
for ticker in tqdm(INDEX_DATA):
    data = INDEX_DATA[ticker]

    data = data[~data.index.duplicated(keep = "last")]

    INDEX_DATA[ticker] = data.copy(deep = True)

100%|█████████████████████████████████████████████████████████████████████████████| 186/186 [00:00<00:00, 11229.41it/s]


In [55]:
ticker

'TPX.GI'

In [61]:
data.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-22,15.3891,2.9469,2717.09
2025-05-23,15.4820,2.9285,2735.52
2025-05-26,15.5696,2.9116,2751.91
2025-05-27,15.6597,2.8948,2769.49
2025-05-28,15.6576,2.8950,2769.51


# Download New Tickers

In [11]:
for ticker, start in tqdm(INDEX_START.items()):
    if ticker in INDEX_DATA:
        continue
        
    assert ticker not in INDEX_DATA
    
    data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]
    
    junk = data['PE_TTM']

    if data.isna().sum().sum() != 0:
        print(ticker)
    
    data.dropna(inplace = True)
    
    INDEX_DATA[ticker] = data.copy(deep = True)

 99%|███████████████████████████████████████▌| 187/189 [00:00<00:00, 270.33it/s]

SP6CSSUP.SPI


100%|█████████████████████████████████████████| 189/189 [00:02<00:00, 84.13it/s]


In [62]:
for ticker, data in INDEX_DATA.items():
    path = os.path.join("Data", f"{ticker}.csv")
    data.to_csv(path, index = True, encoding = "utf-8")

In [63]:
len(INDEX_DATA)

186

In [54]:
INDEX_DATA['000913.SH'].tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-21,28.849001,1.9613,8112.2357
2025-05-22,28.774700,1.9664,8088.3376
2025-05-23,28.756399,2.0349,8082.1061
2025-05-26,28.567101,2.0483,7977.7433
2025-05-27,28.647600,2.0428,8001.2779


In [123]:
temp = INDEX_DATA['000913.SH']


In [124]:
temp.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-09,28.068100,1.9990,7848.4305
2025-05-12,27.993900,1.8358,7818.1952
2025-05-13,28.240000,1.8197,7891.3630
2025-05-14,28.418400,1.8083,7959.5245
2025-05-15,28.409901,1.8086,7968.7475


In [57]:
ticker = '伦敦现货黄金*50%+(MSCI ACWI SELECT GOLD MINERSIMI INDEX)*50%'

temp - w.wsd(ticker,  'pe_ttm,dividendyield2,close', test_start, today, '', usedf = True)[1]

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-22,NaN,NaN,NaN
2025-05-23,NaN,NaN,NaN
2025-05-26,NaN,NaN,NaN
2025-05-27,NaN,NaN,NaN
